<a href="https://colab.research.google.com/github/diseasemodeling/MIE_525_625/blob/main/Lectures/7b_CNN_applications_theory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Applications of CNNs: Classification, Segmentation and Object Detection
## Theoretical background


1. **Problem setup** – what the input and output look like
2. **Architectures** – common designs, with *snapshot code* of just the architecture (PyTorch)
3. **New CNN components** – building blocks specific to some of tehse tasks
4. **Loss function**
5. **Evaluation metrics**
6. **Pretrained models** you can start from


Sources:
* Chapter 14 (and 5.1, 5.7), *Probabilistic Machine Learning: An Introduction*, Kevin Murphy
* *Dive into Deep Learning*, Zhang, Lipton, Li, Smola — chapters 8 (modern CNNs) and 14 (computer vision) https://d2l.ai
* Original papers cited in each section



# 0. **The four tasks at a glance**

Let an image be $\mathbf{x}\in\mathbb{R}^{C\times H\times W}$ (PyTorch uses channels-first, NCHW). The tasks differ in **what we predict**:

| Task | Question answered | Output | Typical output head |
|---|---|---|---|
| Binary classification | *Is there a defect?* | $y\in\{0,1\}$ | 1 logit → sigmoid |
| Multi-class classification | *Which one of $K$ classes?* | $y\in\{1,\dots,K\}$ | $K$ logits → softmax |
| Multi-label classification | *Which of $K$ attributes are present?* | $\mathbf{y}\in\{0,1\}^K$ | $K$ logits → $K$ sigmoids |
| Semantic segmentation | *Which class does every pixel belong to?* | $\mathbf{Y}\in\{1,\dots,K\}^{H\times W}$ | $K$ logits **per pixel** |
| Instance segmentation | *Which object instance does every pixel belong to?* | set of (mask, class) | detector + mask head |
| Object detection | *What objects are where?* | variable-size set $\{(b_j, c_j)\}_{j=1}^{M}$, box $b_j=(x_1,y_1,x_2,y_2)$ | class + box regression per location/anchor |

<img src="https://raw.githubusercontent.com/d2l-ai/d2l-en/master/img/catdog.jpg" width="300">

*Classification asks "cat or dog?"; detection asks "where is the cat and where is the dog?"; segmentation asks "which pixels are cat, which are dog?". Figure source: embedded from d2l.ai (Section 14.3), github.com/d2l-ai/d2l-en.*

## One image, four tasks

An image does not decide the task — **the question we ask about it (how we setup CNN architecture) does.** For an image of a puppy and a kitten:

| Question | Task | Output |
|---|---|---|
| "Is there a dog? Is there a cat?" | **Multi-label classification** | `dog = 1, cat = 1` |
| "Where is each animal?" | **Object detection** | two boxes: *dog* (left), *cat* (right) |
| "Which pixels are dog, which are cat?" | **Semantic segmentation** | a mask with dog pixels, cat pixels, background |
| "Which pixels belong to each individual animal?" | **Instance segmentation** | one mask per animal |


**A common pattern.** Nearly every vision model is

$$\text{image}\;\xrightarrow{\text{backbone (feature extractor)}}\;\text{feature maps}\;\xrightarrow{\text{task-specific head}}\;\text{output}$$

The **backbone** (ResNet, EfficientNet, MobileNet, ConvNeXt, …) is usually shared across tasks and pretrained on ImageNet; the **head** changes with the task. This is why the same pretrained backbone can be reused for classification, segmentation and detection.

---

In [ ]:
# Imports used by the snapshot code cells in this notebook (architecture definitions only; no training here)
import torch
import torch.nn as nn
import torch.nn.functional as F

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# I. **Binary classification**

## I.1 Problem setup
Data $\mathcal{D}=\{(\mathbf{x}_n, y_n)\}_{n=1}^N$, $y_n\in\{0,1\}$ (e.g., defective vs. good part, tumor vs. no tumor, cat vs. dog).

The network computes a single real-valued **logit** $z_n = f_{\boldsymbol\theta}(\mathbf{x}_n)\in\mathbb{R}$, and

$$
\hat p_n = \Pr(y_n=1\mid\mathbf{x}_n;\boldsymbol\theta) = \sigma(z_n) = \frac{1}{1+e^{-z_n}} .
$$

Decision rule: predict $\hat y_n = \mathbb{1}[\hat p_n \ge \tau]$, with threshold $\tau$ (default 0.5; it can be tuned for the cost of false positives vs. false negatives).



## I.2 Architecture
`[conv → BN → ReLU → pool] × L → global average pool → (dropout) → linear(·, 1)`

Any backbone from the previous lecture works. The only change from multi-class is a final linear layer with **one output**.

In [ ]:
# Snapshot: a small CNN for binary classification (architecture only)
class SmallCNN(nn.Module):
    def __init__(self, in_ch=3, n_out=1, width=32, p_drop=0.3):
        super().__init__()
        def block(cin, cout):# define reusable block to stack multiple conv
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1, bias=False),  # 'same' conv; bias is redundant before BN; (3X3) filter
                nn.BatchNorm2d(cout),
                nn.ReLU(inplace=True),#inplace=True overwrites exisiting data instead of creating new tensor -to reduce memory needs
                nn.MaxPool2d(2))                                  # halves H and W
        self.features = nn.Sequential(block(in_ch, width), block(width, 2*width), block(2*width, 4*width))
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),   # global average pooling -> works for any input size
            nn.Flatten(),
            nn.Dropout(p_drop),
            nn.Linear(4*width, n_out)) # n_out=1 -> one logit (binary); n_out=K -> K logits (multi-class)

    def forward(self, x):
        return self.head(self.features(x))  # returns LOGITS (no sigmoid): the loss applies it in a stable way

model = SmallCNN(n_out=1)
print(model(torch.randn(4, 3, 64, 64)).shape, f"{count_params(model):,} parameters")

torch.Size([4, 1]) 93,601 parameters


## I.3 Loss function: binary cross-entropy (BCE)
**Cross-entropy loss**:

(Note: it is also called the **negative log-likelihood** (NLL) as it is equivalent to taking the maximum likelihood of a Bernoulli model $y_n\sim\text{Ber}(\hat p_n)$ which gives the NL:

$$
\mathcal{L}_{\text{BCE}}(\boldsymbol\theta) = -\frac{1}{N}\sum_{n=1}^{N}\Big[y_n\log\hat p_n + (1-y_n)\log(1-\hat p_n)\Big].
$$

*Implementation note:* use `nn.BCEWithLogitsLoss` on logits, which computes $\log\sigma(z)$ with the log-sum-exp trick, rather than `sigmoid` followed by `BCELoss` (numerically unstable when $|z|$ is large).

**Class imbalance** (e.g., 1% defective parts). Two common fixes:

* **Weighted BCE** – up-weight the rare class with $w_1$ (often $w_1 = N_0/N_1$; `pos_weight` in PyTorch):
$$
\mathcal{L}_{\text{wBCE}} = -\frac{1}{N}\sum_n\Big[w_1\,y_n\log\hat p_n + (1-y_n)\log(1-\hat p_n)\Big]
$$
* **Focal loss** (Lin et al., 2017) – down-weight *easy*, well-classified examples so training focuses on hard ones. Let $p_t = \hat p_n$ if $y_n=1$ and $1-\hat p_n$ otherwise:
$$
\mathcal{L}_{\text{focal}} = -\frac{1}{N}\sum_n \alpha_t\,(1-p_t)^{\gamma}\log p_t ,\qquad \gamma\ge 0 \;(\text{typically } 2).
$$
With $\gamma=0$ and $\alpha_t=1$ this is BCE. When an example is easy ($p_t\to 1$), the factor $(1-p_t)^\gamma\to 0$. Focal loss returns in object detection (Section IV).


## I.4 Evaluation metrics
Counts from the **confusion matrix** at threshold $\tau$:

| | predicted 1 | predicted 0 |
|---|---|---|
| **true 1** | TP | FN |
| **true 0** | FP | TN |

$$
\text{Accuracy}=\frac{TP+TN}{TP+TN+FP+FN},\quad
\text{Precision}=\frac{TP}{TP+FP},\quad
\text{Recall (sensitivity, TPR)}=\frac{TP}{TP+FN},
$$
$$
\text{Specificity (TNR)}=\frac{TN}{TN+FP},\qquad \text{FPR} = 1-\text{Specificity},\qquad
F_1=\frac{2\cdot\text{Precision}\cdot\text{Recall}}{\text{Precision}+\text{Recall}},\qquad
F_\beta=\frac{(1+\beta^2)\,P\,R}{\beta^2 P + R}.
$$

Threshold-free metrics (sweep $\tau$ from 1 to 0):
* **ROC curve**: TPR$(\tau)$ vs. FPR$(\tau)$; $\text{AUROC}=\int_0^1 \text{TPR}\;d(\text{FPR})$. Interpretation: AUROC $=\Pr(\hat p(\mathbf{x}^+) > \hat p(\mathbf{x}^-))$ for a random positive $\mathbf{x}^+$ and negative $\mathbf{x}^-$.
* **Precision–Recall curve** and **Average Precision** $\text{AP}=\sum_k (R_k - R_{k-1})P_k$. Prefer PR-AUC/AP over AUROC when positives are rare: AUROC can look excellent even when precision is poor.

**Why not only accuracy?** With 1% defects, a model that always says "good" has 99% accuracy and 0% recall.

---

## I.5 New CNN components

### i) **Data augmentation**
CNNs have millions of parameters; augmentation creates new training samples by applying **label-preserving transformations** $T$ to the input:
$$
\mathcal{L}_{\text{aug}}(\boldsymbol\theta) = \frac{1}{N}\sum_{n}\;\mathbb{E}_{T\sim\mathcal{T}}\big[\ell\big(f_{\boldsymbol\theta}(T(\mathbf{x}_n)),\,y_n\big)\big],
$$
estimated by sampling a fresh random $T$ each time an image is loaded. This injects **prior knowledge about invariances** (a flipped cat is still a cat) and acts as a regularizer.

| Family | Examples | Caution |
|---|---|---|
| Geometric | random crop, horizontal flip, rotation, scale, affine | vertical flip is wrong for digits "6"/"9" or road scenes |
| Photometric | brightness, contrast, saturation, hue jitter, blur, noise | color may carry the label (e.g., ripe vs. unripe fruit) |
| Occlusion | Cutout / RandomErasing | |
| Mixing | **Mixup**, **CutMix** | labels must be mixed too |
| Learned/searched policies | AutoAugment, RandAugment, TrivialAugment | |

**Mixup** (Zhang et al., 2018): sample $\lambda\sim\text{Beta}(\alpha,\alpha)$ and two examples,
$$
\tilde{\mathbf{x}} = \lambda\mathbf{x}_i + (1-\lambda)\mathbf{x}_j,\qquad \tilde{\mathbf{y}} = \lambda\mathbf{y}_i + (1-\lambda)\mathbf{y}_j \;\;(\text{one-hot labels}).
$$
**CutMix** pastes a rectangle of $\mathbf{x}_j$ into $\mathbf{x}_i$ and sets $\lambda$ = fraction of area kept from $\mathbf{x}_i$.

Augmentation is applied **only to training data**. Test data gets only deterministic preprocessing (resize, normalize). *Test-time augmentation (TTA)* averages predictions over several augmented copies of a test image—a separate, optional trick.

In [ ]:
# Snapshot: a typical augmentation pipeline (torchvision.transforms.v2)
from torchvision.transforms import v2 as T

train_tf = T.Compose([
    T.RandomResizedCrop(224, scale=(0.6, 1.0)),    # random scale + aspect + position
    T.RandomHorizontalFlip(p=0.5), # RandomFlip("horizontal")—Applies horizontal flipping to a random 50% of the images that go through it.
                                   #Other options: RandomRotation(0.1)—Rotates the input images by a random value
                                   # andomZoom—Zooms in or out of the image by a random facto
range [-20%, +20%]
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToImage(), T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # ImageNet statistics
    T.RandomErasing(p=0.25),                        # Cutout-style occlusion (acts on tensors)
])
test_tf = T.Compose([T.Resize(256), T.CenterCrop(224),
                     T.ToImage(), T.ToDtype(torch.float32, scale=True),
                     T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

# Mixup / CutMix act on whole *batches* (images and integer labels) -> use after the DataLoader
mix = T.RandomChoice([T.MixUp(num_classes=10, alpha=0.2), T.CutMix(num_classes=10, alpha=1.0)])
xb, yb = mix(torch.rand(8, 3, 224, 224), torch.randint(0, 10, (8,)))
print(xb.shape, yb.shape, "<- labels are now soft (probability vectors)")

torch.Size([8, 3, 224, 224]) torch.Size([8, 10]) <- labels are now soft (probability vectors)


* RandomFlip("horizontal")—Applies horizontal flipping to a random 50% of the images that go through it.
* RandomRotation(0.1)—Rotates the input images by a random value in the range [–10%, +10%] (these are fractions of a full circle—in degrees, the range would be [–36 degrees, +36 degrees])
* RandomZoom(0.2)—Zooms in or out of the image by a random factor in the
range [-20%, +20%]

### ii) **Transfer learning: feature extraction vs. fine-tuning**
Collecting thousands of labeled images is expensive. Instead start from a backbone $g_{\boldsymbol\phi}$ pretrained on a large dataset (ImageNet-1k: 1.28M images, 1000 classes) and attach a new head $h_{\boldsymbol\psi}$:
$$
f(\mathbf{x}) = h_{\boldsymbol\psi}\big(g_{\boldsymbol\phi}(\mathbf{x})\big)
$$

* **Feature extraction:** freeze $\boldsymbol\phi$ (`requires_grad=False`), train only $\boldsymbol\psi$. Fast; good when the new dataset is small and similar to ImageNet.
* **Fine-tuning:** train $\boldsymbol\psi$ and (part of) $\boldsymbol\phi$, often with a smaller learning rate for $\boldsymbol\phi$ ("discriminative learning rates"), e.g., $\eta_{\boldsymbol\phi}=\eta_{\boldsymbol\psi}/10$. Better when data is larger or differs from ImageNet (medical, satellite, microscopy).

Why it works: early layers learn generic features (edges, color blobs, textures), later layers learn task-specific parts; generic features transfer.

**Practical rule:** the input must be preprocessed the way the backbone was trained (same resize and the ImageNet mean/std normalization).

In [ ]:
# Snapshot: transfer learning with a torchvision ResNet-18 (weights=None here to keep this cell offline;
# use weights=ResNet18_Weights.DEFAULT in practice to download ImageNet weights)
from torchvision.models import resnet18, ResNet18_Weights

backbone = resnet18(weights=None)                 # resnet18(weights=ResNet18_Weights.DEFAULT)
for p in backbone.parameters():                   # 1) feature extraction: freeze everything ...
    p.requires_grad = False
backbone.fc = nn.Linear(backbone.fc.in_features, 1)  # ... 2) new trainable head (1 logit for binary classification)# replaces the existing fully connected (fc) layer at the end of the ResNet model with a brand new, un-trained linear layer
print("trainable params (head only):", count_params(backbone))

# 3) fine-tuning: unfreeze the last residual stage and use a smaller learning rate for it
for p in backbone.layer4.parameters():
    p.requires_grad = True
optimizer = torch.optim.AdamW([
    {"params": backbone.fc.parameters(),     "lr": 1e-3},
    {"params": backbone.layer4.parameters(), "lr": 1e-4}])
print("trainable params (head + layer4):", count_params(backbone))

trainable params (head only): 513
trainable params (head + layer4): 8394241


In [ ]:
print(backbone)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

### iii) **Regularization extras: dropout and label smoothing**
* **Dropout** before the final linear layer: each feature is zeroed with probability $p$ during training (and the rest scaled by $1/(1-p)$), preventing co-adaptation of features.
* **Label smoothing** (see multi-class section) and **weight decay** ($\lambda\|\boldsymbol\theta\|_2^2$, decoupled in AdamW) are the other common regularizers.

---

## I.7 Pretrained models for classification
All of the following have ImageNet-1k weights in `torchvision.models` (`weights=...DEFAULT`):

| Model | Year | Key idea | Params | Note |
|---|---|---|---|---|
| VGG-16 | 2014 | stacks of 3×3 convs | 138M | simple but heavy |
| ResNet-18/50 | 2015 | residual connections | 11.7M / 25.6M | **default first choice** |
| DenseNet-121 | 2017 | dense concatenation | 8M | |
| MobileNetV2/V3 | 2018/19 | depthwise separable convs, inverted residuals | 2–5M | phones, edge devices |
| EfficientNet-B0…B7 | 2019 | compound scaling | 5.3M (B0) | accuracy per FLOP |
| RegNet | 2020 | design-space search | various | |
| ConvNeXt-T | 2022 | CNN modernized with Transformer design choices | 28M | strong modern CNN |
| ViT-B/16, Swin | 2020/21 | Vision Transformers (non-CNN) | 86M | need more data / strong pretraining |

More weights: `timm` (PyTorch Image Models, hundreds of backbones), Hugging Face Hub, `keras.applications` (Keras). Domain-specific starting points exist too, e.g., RadImageNet (radiology), BigEarthNet / SSL4EO models (satellite imagery).

---

# II. **Multi-class (and multi-label) classification**



## II.1 Problem setup
$y_n\in\{1,\dots,K\}$, one-hot encoded as $\mathbf{y}_n\in\{0,1\}^K$ with $\sum_k y_{nk}=1$ (e.g., CIFAR-10, ImageNet, defect *type*). The network outputs $K$ logits $\mathbf{z}_n\in\mathbb{R}^K$, and the **softmax** turns them into a categorical distribution:
$$
\hat p_{nk} = \Pr(y_n = k\mid\mathbf{x}_n) = \text{softmax}(\mathbf{z}_n)_k=\frac{e^{z_{nk}}}{\sum_{j=1}^K e^{z_{nj}}},\qquad \hat y_n = \arg\max_k \hat p_{nk}.
$$
**Multi-label** (several labels can be true at once, e.g., an X-ray showing both effusion and cardiomegaly): use $K$ *independent* sigmoids, $\hat p_{nk}=\sigma(z_{nk})$, and a per-class threshold.



## II.2 Architecture
Same backbones; the head is `linear(·, K)`. Below we add three components that define *modern efficient* CNNs: depthwise separable convolution, inverted residual blocks, and squeeze-and-excitation.

## II.3 New CNN components (not specific to classifictaion just general for as tasks get more complicated)



### i) **Depthwise separable convolution** (Xception, MobileNet)
* Standard convolution uses a filter of size $H × W × C × D$, (height, width, channels, depth) which requires a lot of data to learn and a lot of time to compute with.
* Depthwise separable convolution, first convolves each input channel by a corresponding 2d filter w, and then maps these C channels to D channels using $1 × 1$ convolution $w′$
$$
z_{i,j,d} = b_d +
\sum_{c=0}^{C-1}
w'_{c,d}
\sum_{u=0}^{H-1}
\sum_{v=0}^{W-1}
x_{i + u,\, j + v,\, c} \, w_{u,v}
$$


In PyTorch, depthwise convolution is `nn.Conv2d(C, C, 3, groups=C)`. (`groups=g` splits channels into $g$ groups that are convolved separately: **grouped convolution**, used in AlexNet and ResNeXt; `groups=C` is the extreme case.)

<img src="https://github.com/probml/pml-book/blob/main/book1-figures/Figure_14.26.png?raw=true" height="250">

*Figure: Depthwise separable convolution: a 2d convolution per channel, followed by a 1×1 convolution that maps $C$ to $D$ channels. Source: Embedded from Github repo of textbook (PML: An Introduction by Kevin Murphy). See textbook for original source.*

* It is more efficient
  * Regular convolution of a $12 × 12 × 3$ input with a $5 × 5 × 3 × 256$ filter gives a $8 × 8 × 256$ output (assuming valid convolution:12-5+1=8).
  * With separable convolution, we start with $12×12×3$ input, convolve with a $5 × 5 × 1 × 1$ filter (across space but not channels) to get $8 × 8 × 3$, then pointwise convolve (across channels but not space) with a $1 × 1 × 3 × 256$ filter to get a $8 × 8 × 256$ output. So the output has the same size as before, but we used many fewer parameters to define the layer, and used much less compute.
* The efficiency of these convolutions makes them suitable for real-time applications like image classification and object detection on mobile devices and embedded systems, such as drones.

Cons: can reduce representational power vs a full conv and may require more layers or careful tuning to match accuracy.

In [ ]:
# Snapshot: depthwise separable convolution vs. standard convolution
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, cin, cout, k=3, stride=1):
        super().__init__()
        self.depthwise = nn.Conv2d(cin, cin, k, stride, padding=k//2, groups=cin, bias=False)  # spatial, per channel
        self.pointwise = nn.Conv2d(cin, cout, 1, bias=False)                                   # channel mixing
        self.bn1, self.bn2 = nn.BatchNorm2d(cin), nn.BatchNorm2d(cout)
    def forward(self, x):
        x = F.relu(self.bn1(self.depthwise(x)))
        return F.relu(self.bn2(self.pointwise(x)))

std = nn.Conv2d(128, 256, 3, padding=1, bias=False)
sep = DepthwiseSeparableConv(128, 256)
x = torch.randn(1, 128, 32, 32)
print("output shapes:", std(x).shape, sep(x).shape)
print(f"standard: {count_params(std):,} params | separable: {count_params(sep):,} params "
      f"(ratio {count_params(sep)/count_params(std):.3f}; theory 1/256+1/9 = {1/256+1/9:.3f}, BN adds a few params)")

## II.4 Loss function: categorical cross-entropy

**Multi-class**: catergorical cross-entropy (negative log-likelihood of the categorical distribution):
$$
\mathcal{L}_{\text{CE}}=-\frac{1}{N}\sum_{n=1}^N\sum_{k=1}^K y_{nk}\log\hat p_{nk}$$
$K$ is number of classes; $N$ is number of samples (images)

Variants: class-weighted CE ($w_{y_n}$ multiplies each term), CE with label smoothing (replace $y_{nk}$ by $\tilde y_{nk}$), focal loss with $p_t=\hat p_{n,y_n}$.

**Multi-label:** sum of $K$ binary cross-entropies:
$\;\mathcal{L}=-\frac1N\sum_n\sum_k\big[y_{nk}\log\sigma(z_{nk})+(1-y_{nk})\log(1-\sigma(z_{nk}))\big]$.

## II.5 Evaluation metrics
With $K$ classes, a single number such as accuracy hides most of what we want to know about a classifier. Accuracy counts how often the top prediction is right, but it says nothing about *which* classes are confused with each other, whether rare classes are recognized at all, or whether the predicted probabilities can be trusted. The metrics below look at the model from these different angles: **top-$k$ accuracy** asks whether the right answer is at least among the model's best guesses; the **confusion matrix** shows the pattern of mistakes; **per-class precision, recall and $F_1$**, averaged in different ways, reveal whether good overall accuracy comes at the expense of small classes; and **calibration** checks whether a confidence of 0.8 really means "right 80% of the time", which matters whenever the probability itself drives a decision (e.g., sending uncertain cases to a human expert). Most of these extend the binary metrics from previous section by treating each class in turn as the "positive" class (one-vs-rest).
* **Top-1 accuracy** $=\frac1N\sum_n\mathbb{1}[\hat y_n=y_n]$; **top-$k$ accuracy** $=\frac1N\sum_n\mathbb{1}[y_n\in\text{top-}k(\hat{\mathbf p}_n)]$ (ImageNet reports top-5).
* **$K\times K$ confusion matrix** $M_{ab}=\#\{n: y_n=a,\hat y_n=b\}$; shows *which* classes get confused (e.g., cat ↔ dog).
* **Per-class (one-vs-rest)** precision/recall: $P_k = \frac{M_{kk}}{\sum_a M_{ak}}$ (column sum), $R_k = \frac{M_{kk}}{\sum_b M_{kb}}$ (row sum), $F_{1,k}=\frac{2P_kR_k}{P_k+R_k}$.
* **Averaging** over classes:
  * **macro**: $F_1^{\text{macro}}=\frac1K\sum_k F_{1,k}$ — every class counts equally (good for imbalanced data);
  * **weighted**: $\sum_k \frac{N_k}{N}F_{1,k}$;
  * **micro**: pool TP/FP/FN over classes first; for single-label problems micro-$F_1$ = accuracy.

* **Calibration**: are 80%-confident predictions right 80% of the time?
  Let the confidence of prediction of image $n$ be its largest predicted probability, $\text{conf}_n=\max_k \hat p_{nk}$. Sort the $N$ test samples into $M$ equal-width confidence bins $B_1,\dots,B_M$ (e.g., $[0,0.1), [0.1,0.2), \dots$) and, in each bin, compare the average confidence with the fraction of correct predictions:
  $$
  \text{conf}(B_m)=\frac{1}{|B_m|}\sum_{n\in B_m}\text{conf}_n,\qquad
  \text{acc}(B_m)=\frac{1}{|B_m|}\sum_{n\in B_m}\mathbb{1}[\hat y_n=y_n],\qquad
  \text{ECE}=\sum_{m=1}^{M} \frac{|B_m|}{N}\,\big|\text{acc}(B_m)-\text{conf}(B_m)\big| .
  $$
  ECE is a weighted average gap between confidence and accuracy ($0$ = perfectly calibrated). Plotting $\text{acc}(B_m)$ against $\text{conf}(B_m)$ gives the **reliability diagram**; a calibrated model lies on the diagonal, while points *below* the diagonal mean **over-confidence** (e.g., predictions made with 95% confidence are right only 80% of the time). Modern deep networks are often over-confident (Guo et al., 2017), because training with cross-entropy keeps pushing the correct-class probability towards 1. Accuracy does not reveal this, yet it matters whenever the probability is used for a decision, such as flagging uncertain cases for human review. Common remedies are **label smoothing**  and **temperature scaling**: after training, replace $\text{softmax}(\mathbf z)$ by $\text{softmax}(\mathbf z/T)$ (where $z$ are the logit scores) with a single scalar $T>0$ fitted on the validation set by minimizing the negative log-likelihood. $T>1$ softens over-confident predictions without changing which class is predicted, so accuracy is unchanged.

* **Multi-label metrics**: each image can have several true labels, $\mathbf y_n\in\{0,1\}^K$, and the model outputs an independent probability per class, $\hat p_{nk}=\sigma(z_{nk})$. Metrics are therefore computed **per class** and then averaged:
  * **Average precision (AP) and mAP** (threshold-free): for class $k$, rank all test images by $\hat p_{nk}$ and walk down the list; after the top $r$ images compute precision $P_r$ and recall $R_r$ for that class, and summarize the precision–recall curve by $\text{AP}_k=\sum_r (R_r-R_{r-1})\,P_r$. The mean over classes, $\text{mAP}=\frac1K\sum_k \text{AP}_k$, is the standard benchmark metric (e.g., PASCAL VOC and COCO classification). It measures how well the model *ranks* images containing class $k$ above those that do not, independently of any threshold.
  * **$F_1$ at chosen thresholds** (decision quality): to produce actual labels, predict class $k$ present if $\hat p_{nk}\ge\tau_k$. Thresholds are chosen per class on the validation set, because rare classes often need lower thresholds. Counting $TP_k, FP_k, FN_k$ for each class then gives per-class precision, recall and $F_{1,k}$. **Macro-$F_1$** $=\frac1K\sum_k F_{1,k}$ weights every class equally, so rare labels count as much as common ones. **Micro-$F_1$** pools the counts first, $\;\frac{2\sum_k TP_k}{2\sum_k TP_k+\sum_k FP_k+\sum_k FN_k}$, so frequent labels dominate. Unlike in the single-label case, micro-$F_1$ is *not* equal to accuracy here.
  * Per-image ("example-based") metrics are also reported. **Subset accuracy**, the fraction of images whose entire label vector is predicted exactly, is very strict. **Hamming loss** $=\frac{1}{NK}\sum_{n,k}\mathbb 1[\hat y_{nk}\ne y_{nk}]$ is the fraction of individual labels that are wrong.

---

## II.6 Pretrained models
Same as in binary classification. For a $K$-class problem replace the final layer, e.g., `model.fc = nn.Linear(512, K)` (ResNet) or `model.classifier[-1] = nn.Linear(1280, K)` (EfficientNet/MobileNetV2). The ImageNet classes themselves can be used zero-shot if your classes are among them. **Zero-shot** with arbitrary text labels is possible with vision–language models such as CLIP (`open_clip`), which score an image against prompts like "a photo of a {class}".

---

# III. **Image segmentation**


## III.1 Problem setup

Image segmentation assigns a class to each pixel in an image, thus segmenting the image into different zones (such as “background” and “foreground” or “road,” “car,” and “sidewalk”).

  * A **segmentation mask** is the image segmentation equivalent of a label: it’s an image the same size as the input image, with a single color channel where each integer value corresponds to the class of the corresponding pixel in the input image.

Segmentation types
* **Semantic segmentation**: assign a class to every pixel; $\mathbf{Y}\in\{1,\dots,K\}^{H\times W}$. Two touching cars are both "car".
* **Instance segmentation**: also separate object instances (car #1, car #2) → a set of binary masks with classes.
* **Panoptic segmentation**: semantic for "stuff" (sky, road) + instances for "things" (cars, people).

Applications: medical imaging (tumor/organ boundaries), autonomous driving (drivable road), manufacturing (crack and defect area), remote sensing (land cover), microscopy (cell counting).

<img src="https://raw.githubusercontent.com/d2l-ai/d2l-en/master/img/segmentation.svg" width="500">

*Figure: Semantic segmentation labels each pixel (dog, cat, background). Source: embedded from d2l.ai (Section 14.9).*

For each pixel $(i,j)$ the network outputs logits $\mathbf{z}_{ij}\in\mathbb{R}^K$ and $\hat{\mathbf p}_{ij}=\text{softmax}(\mathbf{z}_{ij})$. So the output tensor is $K\times H\times W$: **segmentation = classification at every pixel**.

**The central difficulty:** classification CNNs *downsample* (a $224\times224$ image becomes a $7\times7$ feature map) to gain a large receptive field and semantic features, but segmentation needs a **full-resolution output**. Architectures differ in how they recover resolution while keeping context.

| Setting | What labels are used | Examples |
|---|---|---|
| **Supervised** | full pixel-level masks | U-Net, DeepLab, Mask R-CNN |
| **Weakly supervised** | cheaper labels: image-level tags, boxes, scribbles, clicks | CAM/Grad-CAM maps turned into pseudo-masks; box-supervised methods |
| **Promptable / foundation models** | trained supervised on huge mask datasets, then used without task-specific labels | Segment Anything (SAM): you click or draw a box, it returns a mask |

* A common way to tackle semantic segmentation is to use an **encoder–decoder** architecture (e.g., FCN, SegNet, U-Net, DeepLabV3+).
  * The **encoder** is usually a classification CNN (often pretrained, e.g., VGG or ResNet) whose convolution and pooling layers map the input to a low-resolution feature map (the *bottleneck*). It has few spatial positions but many channels (features), capturing high-level, semantic properties ("what is in the image") at a coarse spatial resolution. Some models (DeepLab) replace the last downsampling steps with **dilated convolution**, which enlarges the receptive field (more context) while keeping a larger feature map.
  * The **decoder** maps the bottleneck back to a full-size output with one score per class at every pixel. Models differ in how they upsample: **transposed convolution** (FCN, U-Net), **max-unpooling** using the pooling positions stored by the encoder (SegNet), or **bilinear interpolation** (DeepLab).
  * Because downsampling loses precise location information ("where exactly is the boundary"), many models add **skip connections** that pass encoder feature maps to the decoder layer of the same resolution. FCN *adds* them, U-Net *concatenates* them, and SegNet passes only the pooling indices.
* **Instance and panoptic segmentation** still use an encoder–decoder at their core, but a semantic encoder–decoder alone outputs only *one class per pixel*, so two touching objects of the same class merge into one region. These tasks add a mechanism to **separate objects**:
  * **Detect, then segment** (Mask R-CNN): a detector proposes one box per object, and a small FCN mask head predicts a binary mask inside each box. Its FPN backbone is itself an encoder–decoder (top-down pathway with lateral skip connections).
  * **Segment, then group** (Panoptic-DeepLab): an encoder–decoder predicts, for every pixel, its class *and* the offset to its object's centre; pixels pointing to the same centre are grouped into one instance.
  * (Tranformer based vision models are more common now - just a ping for tranformers here ; below we will stick to CNN) **Mask queries** (Mask2Former): a pixel decoder produces per-pixel embeddings, and a transformer turns $N$ learned queries into $N$ (class, mask) predictions. A single model handles semantic, instance and panoptic segmentation.
* **Panoptic segmentation** combines the two: every pixel gets a class (including "stuff" such as sky or road), and pixels of countable "things" (cars, people) also get an instance ID.

We will first discuss these new CNN components below

## III.2 New CNN components



### i) **Transposed convolution** ("deconvolution", learnable upsampling)
A convolution maps many input pixels to one output pixel. A **transposed convolution** maps one input pixel to many outputs: each input value $x_{ij}$ *scales the whole kernel* $\mathbf{W}$, the scaled kernel is placed at position $(s\,i, s\,j)$ in the output, and overlapping contributions are summed:
$$
y_{s i + u,\; s j + v}\mathrel{+}= x_{i,j}\,w_{u,v},\qquad u=0..f_h-1,\; v=0..f_w-1 .
$$
Why "transposed": write a convolution as a matrix product $\mathbf{y}=\mathbf{C}\mathbf{x}$ on flattened images; the transposed convolution computes $\mathbf{C}^\top\mathbf{y}$, mapping the *output shape back to the input shape* (it is not the mathematical inverse). Output size (per dimension, padding $p$, stride $s$):
$$
n_{\text{out}} = (n_{\text{in}}-1)\,s - 2p + f .
$$
With $f=2, s=2, p=0$: $n_{\text{out}}=2n_{\text{in}}$ (exact doubling, the U-Net choice).

<img src="https://raw.githubusercontent.com/d2l-ai/d2l-en/master/img/trans_conv.svg" width="450">

*Figure: Transposed convolution with a 2×2 kernel, stride 1. Each input element multiplies the kernel and the results are summed in the output. Source: embedded from d2l.ai (Section 14.10).*

**Checkerboard artifacts:** when $f$ is not divisible by $s$, output pixels receive uneven numbers of contributions. Alternative: **resize-convolution** = bilinear upsampling followed by a $3\times3$ conv (`nn.Upsample` + `nn.Conv2d`).

In [ ]:
# Snapshot: transposed convolution from scratch vs. nn.ConvTranspose2d
def trans_conv(X, K, stride=1):
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - 1) * stride + h, (X.shape[1] - 1) * stride + w)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            Y[i*stride:i*stride + h, j*stride:j*stride + w] += X[i, j] * K   # each input pixel "stamps" the kernel
    return Y

X = torch.tensor([[0., 1.], [2., 3.]]); K = torch.tensor([[0., 1.], [2., 3.]])
print(trans_conv(X, K))                                   # d2l example: 3x3 output
tconv = nn.ConvTranspose2d(1, 1, kernel_size=2, bias=False); tconv.weight.data = K.reshape(1, 1, 2, 2)
print(tconv(X.reshape(1, 1, 2, 2)).squeeze().detach())   # same result
up = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)  # the U-Net upsampler: doubles H, W
print("upsample 16x16 ->", tuple(up(torch.randn(1, 64, 16, 16)).shape[-2:]))

### ii) **Dilated (atrous) convolution** (Yu & Koltun, 2016; DeepLab)
Convolution is an operation that combines the pixel values in a local neighborhood. By using striding, and stacking many layers of convolution together, we can enlarge the receptive field of each neuron, which is the region of input space that each neuron responds to. However, we would need many layers to give each neuron enough context to cover the entire image (unless we used very large filters, which would be slow and require too many parameters).
Alternative is  **convolution with holes**, also known by the French term **à trous algorithm**, and recently renamed **dilated convolution**. This method simply
takes every r’th input element when performing convolution, where $r$ is the rate or dilation factor

In 1D convolution, convolving with filter $w$ using rate $r = 2$ would be equivalent to regular convolution of $w=[w_1,0,w_2,0,w_3]$

In 2D convolution

$$
z_{i,j,d} = b_d +
\sum_{u=0}^{H-1}
\sum_{v=0}^{W-1}
\sum_{c=0}^{C-1}
x_{i + r u,\, j + r v,\, c} \, w_{u,v,c,d}
$$

A $3\times3$ kernel with $r=2$ covers a $5\times5$ region; the effective kernel size is $f_{\text{eff}} = f + (f-1)(r-1)$. Benefits: **larger receptive field with no extra parameters and no downsampling**, so the feature map keeps its resolution. Stacking dilations $1,2,4,8$ grows the receptive field exponentially with depth (instead of linearly). Pitfall: "gridding" artifacts if all layers use the same large $r$.

<img src="https://github.com/probml/pml-book/blob/main/book1-figures/Figure_14.23.png?raw=true" height="200">

*Figure: Dilated convolution with a 3×3 filter using rates 1, 2 and 3. Source: Embedded from Github repo of textbook (PML: An Introduction by Kevin Murphy). See textbook for original source.*

### iii) **Atrous Spatial Pyramid Pooling (ASPP)** (DeepLabv3)
Objects appear at many scales. Apply in **parallel** a $1\times1$ conv, three $3\times3$ dilated convs ($r=6,12,18$) and image-level global pooling; concatenate and fuse with a $1\times1$ conv.

In [ ]:
# Snapshot: dilated convolution keeps resolution while enlarging the receptive field; ASPP module
x = torch.randn(1, 16, 64, 64)
for r in [1, 2, 4]:
    conv = nn.Conv2d(16, 16, 3, padding=r, dilation=r)   # padding=r keeps H, W unchanged for a 3x3 kernel
    print(f"dilation {r}: out {tuple(conv(x).shape[-2:])}, effective kernel {3 + 2*(r-1)}x{3 + 2*(r-1)}, params {count_params(conv)}")

class ASPP(nn.Module):
    def __init__(self, cin, cout=64, rates=(6, 12, 18)):
        super().__init__()
        def cbr(k, r):
            return nn.Sequential(nn.Conv2d(cin, cout, k, padding=0 if k == 1 else r, dilation=r, bias=False),
                                 nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
        self.branches = nn.ModuleList([cbr(1, 1)] + [cbr(3, r) for r in rates])
        self.image_pool = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(cin, cout, 1), nn.ReLU(inplace=True))
        self.project = nn.Sequential(nn.Conv2d(cout * (len(rates) + 2), cout, 1, bias=False),
                                     nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
    def forward(self, x):
        feats = [b(x) for b in self.branches]
        feats.append(F.interpolate(self.image_pool(x), size=x.shape[-2:], mode="bilinear", align_corners=False))
        return self.project(torch.cat(feats, dim=1))

print("ASPP:", ASPP(256)(torch.randn(2, 256, 32, 32)).shape)

dilation 1: out (64, 64), effective kernel 3x3, params 2320
dilation 2: out (64, 64), effective kernel 5x5, params 2320
dilation 4: out (64, 64), effective kernel 9x9, params 2320
ASPP: torch.Size([2, 64, 32, 32])


### iv) **Skip connections by concatenation (encoder → decoder)**
Deep encoder features know **what** is present but have lost **where** exactly. Shallow encoder features have precise locations but little semantics. U-Net concatenates (channel-wise) the encoder map at each resolution with the upsampled decoder map, so the decoder can combine both: $\;\mathbf{D}_\ell = \text{conv}\big([\,\text{up}(\mathbf{D}_{\ell+1}),\;\mathbf{E}_\ell\,]\big)$. (Compare: ResNet *adds* skips, DenseNet *concatenates*.)

### v) **Data augmentation for segmentation**
Geometric transforms must be applied **identically to the image and the mask** (same crop, flip, rotation). Interpolate masks with **nearest neighbor** (bilinear interpolation would create meaningless "in-between" class labels such as 1.5). Photometric transforms are applied to the image only. `torchvision.transforms.v2` does this automatically when the mask is wrapped as `tv_tensors.Mask`; the library `albumentations` is also popular.

---

## III.3 Architectures

### i) **Fully Convolutional Network (FCN)** (Long, Shelhamer, Darrell, 2015)

Suppose we have below for classfication
$$\text{image } (3\times224\times224) \xrightarrow{\text{conv layers}} \text{feature map } \mathbf A\ (512\times7\times7) \xrightarrow{\text{pool / flatten}} \text{vector} \xrightarrow{\text{fully connected}} K \text{ class scores}$$

For segmentation, we replace the fully connected layers by $1\times1$ convolutions (so the network accepts any input size and outputs a $K$-channel *map*), then upsample the coarse map to input resolution with transposed convolution.
$$\mathbf A\ (512\times h\times w) \xrightarrow{1\times1\ \text{conv},\ K \text{ filters}} \mathbf Z\ (K\times h\times w)$$


<img src="https://raw.githubusercontent.com/d2l-ai/d2l-en/master/img/fcn.svg" width="350">

*Figure: Fully convolutional network. Source: embedded from d2l.ai (Section 14.11).*

### ii) **U-Net** (Ronneberger, Fischer, Brox, 2015)
Designed for biomedical images with very few training samples. A symmetric **encoder** (contracting path: conv–conv–pool, channels double, resolution halves) and **decoder** (expanding path: transposed conv, channels halve, resolution doubles) linked by **concatenation skip connections** at every level. Still the default in medical imaging and the denoising network inside diffusion models.

<img src="https://github.com/probml/pml-book/blob/main/book1-figures/Figure_14.30.png?raw=true" height="600">

*Figure: U-Net architecture. Source: Embedded from Github repo of textbook (PML: An Introduction by Kevin Murphy). See textbook for original source.*

### iii) **SegNet** (2017) – encoder–decoder where the decoder upsamples using the *max-pooling indices* stored by the encoder (`nn.MaxUnpool2d`), so no upsampling weights are learned.

<img src="https://github.com/probml/pml-book/blob/main/book1-figures/Figure_14.29.png?raw=true" height="200">

*Figure: SegNet-style convolutional encoder–decoder for semantic segmentation. Source: Embedded from Github repo of textbook (PML: An Introduction by Kevin Murphy). See textbook for original source.*

### iv) **DeepLab v3 / v3+** (Chen et al., 2017, 2018) – ResNet backbone with the last strides replaced by dilation (output stride 8 or 16 instead of 32), ASPP head for multi-scale context; v3+ adds a light decoder that fuses low-level features for sharp boundaries.

### v) **Instance segmentation: Mask R-CNN** (He et al., 2017) – Faster R-CNN detector (Section IV) plus a small FCN head that predicts a $28\times28$ binary mask *inside each detected box*. Uses **RoIAlign** (bilinear sampling) instead of RoIPool for pixel-accurate alignment.

<img src="https://github.com/probml/pml-book/blob/main/book1-figures/Figure_14.28.png?raw=true" height="250">

*Figure: Example output of Mask R-CNN (boxes, classes and instance masks). Source: Embedded from Github repo of textbook (PML: An Introduction by Kevin Murphy). See textbook for original source.*

### vi) **Foundation models: Segment Anything (SAM, SAM 2)** (Kirillov et al., 2023; Ravi et al., 2024) – a ViT image encoder + prompt encoder + mask decoder trained on 1B masks. Given a *prompt* (a click, a box) it returns a mask for any object, without class labels. Useful for fast annotation and zero-shot masks; it is not a CNN and does not name the class.

In [ ]:
# Snapshot: a compact U-Net (architecture only). Input H, W must be divisible by 2**depth.
class DoubleConv(nn.Sequential):
    def __init__(self, cin, cout):
        super().__init__(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                         nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

class UNet(nn.Module):
    def __init__(self, in_ch=3, n_classes=3, base=32, depth=3):
        super().__init__()
        chs = [base * 2**i for i in range(depth + 1)]                    # e.g. 32, 64, 128, 256
        self.downs = nn.ModuleList([DoubleConv(in_ch, chs[0])] +
                                   [DoubleConv(chs[i], chs[i+1]) for i in range(depth)])
        self.pool = nn.MaxPool2d(2)
        self.ups = nn.ModuleList([nn.ConvTranspose2d(chs[i+1], chs[i], 2, stride=2) for i in reversed(range(depth))])
        self.dec = nn.ModuleList([DoubleConv(2 * chs[i], chs[i]) for i in reversed(range(depth))])
        self.out = nn.Conv2d(chs[0], n_classes, 1)                       # per-pixel class logits

    def forward(self, x):
        skips = []
        for i, down in enumerate(self.downs):          # encoder (contracting path)
            x = down(x if i == 0 else self.pool(x))
            skips.append(x)
        x = skips.pop()                                # bottleneck
        for up, dec in zip(self.ups, self.dec):        # decoder (expanding path)
            x = up(x)                                  # transposed conv: x2 resolution
            x = dec(torch.cat([x, skips.pop()], 1))    # concatenate the skip connection
        return self.out(x)                             # (N, K, H, W)

unet = UNet()
print(unet(torch.randn(2, 3, 128, 128)).shape, f"{count_params(unet):,} parameters")

torch.Size([2, 3, 128, 128]) 1,927,075 parameters


In [ ]:
print(unet)

UNet(
  (downs): ModuleList(
    (0): DoubleConv(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
    (1): DoubleConv(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
    (2): DoubleConv(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1,

## III.4 Loss functions
Let $\hat p_{ik}$ be the predicted probability of class $k$ at pixel $i$ (pixels flattened, $i=1..P$), $y_{ik}$ the one-hot ground truth.

* **Pixel-wise cross-entropy** (average of CE over all pixels):
$$
\mathcal{L}_{\text{CE}}=-\frac{1}{P}\sum_{i=1}^{P}\sum_{k=1}^{K} w_k\,y_{ik}\log\hat p_{ik},
$$
with optional class weights $w_k$ (often $\propto 1/\text{frequency}$ or median-frequency balancing) because background usually dominates. Pixels with an "ignore" label (e.g., ambiguous borders, `ignore_index=255`) are excluded.
* **Dice loss** (Milletari et al., 2016) – directly optimizes overlap, robust to class imbalance:
$$
\mathcal{L}_{\text{Dice}} = 1-\frac{1}{K}\sum_{k=1}^{K}\frac{2\sum_i \hat p_{ik}\,y_{ik}+\epsilon}{\sum_i \hat p_{ik}+\sum_i y_{ik}+\epsilon}
$$
("soft" Dice: uses probabilities, so it is differentiable). $\epsilon$ avoids $0/0$ for classes absent from the image.
* **Soft Jaccard / IoU loss**: $\;1-\frac{\sum_i \hat p_{ik}y_{ik}}{\sum_i(\hat p_{ik}+y_{ik}-\hat p_{ik}y_{ik})}$; **Lovász-softmax** is a convex surrogate that optimizes IoU directly.
* **Focal loss** per pixel, and **Tversky loss** ($\frac{TP}{TP+\alpha FP+\beta FN}$, trades off FP vs. FN).
* In practice: **CE + Dice** combined, $\mathcal{L}=\mathcal{L}_{\text{CE}}+\lambda\mathcal{L}_{\text{Dice}}$ (CE gives stable gradients early; Dice focuses on overlap of small structures).

## III.5 Evaluation metrics
Build a $K\times K$ confusion matrix $M$ over **all pixels of all test images** ($M_{ab}$ = # pixels of true class $a$ predicted as $b$).
* **Pixel accuracy** $=\frac{\sum_k M_{kk}}{\sum_{a,b}M_{ab}}$ — misleading when background dominates.
* **IoU (Jaccard index)** of class $k$: Intersection over Union (IoU) is a measure of the match between the ground truth segmentation masks and the predicted masks. It can be computed separately for each class or averaged over multiple classes. it works as follow:

1. Compute the intersection between the masks, the area where the prediction and ground truth overlap.
2.  Compute the union of the masks, the total area covered by both masks combined. This is the whole space we’re interested in — the target object and any extra bits your model might have included by mistake.
3. Divide the intersection area by the union area to get the IoU. It’s a number between 0 and 1, where 1 denotes a perfect match, and 0 denotes a complete miss.

We can simply use a built-in Keras metric rather than building this ourselves:
$$
\text{IoU}_k=\frac{|\hat{S}_k\cap S_k|}{|\hat{S}_k\cup S_k|}=\frac{TP_k}{TP_k+FP_k+FN_k}=\frac{M_{kk}}{\sum_b M_{kb}+\sum_a M_{ak}-M_{kk}},
$$
where $S_k$/$\hat S_k$ are the true/predicted pixel sets of class $k$. **mean IoU (mIoU)** $=\frac1K\sum_k\text{IoU}_k$ is the standard benchmark metric (PASCAL VOC, Cityscapes, ADE20K).

* **Dice coefficient (= pixel-level $F_1$)**: $\;\text{Dice}_k=\frac{2|\hat S_k\cap S_k|}{|\hat S_k|+|S_k|}=\frac{2TP_k}{2TP_k+FP_k+FN_k}$; monotone in IoU: $\text{Dice}=\frac{2\,\text{IoU}}{1+\text{IoU}}$. Standard in medical imaging.
* **Boundary metrics**: Boundary-F1, **Hausdorff distance** $d_H(A,B)=\max\{\sup_{a\in A}\inf_{b\in B}\|a-b\|,\;\sup_{b\in B}\inf_{a\in A}\|a-b\|\}$ (often its 95th percentile, HD95) measure contour accuracy, which region overlap ignores.
* Instance segmentation uses **mask AP** (Section IV metrics, with mask IoU instead of box IoU).

## III.6 Pretrained models
| Source | Models | Pretrained on |
|---|---|---|
| `torchvision.models.segmentation` | `fcn_resnet50/101`, `deeplabv3_resnet50/101`, `deeplabv3_mobilenet_v3_large`, `lraspp_mobilenet_v3_large` | COCO subset with the 20 PASCAL VOC classes (+ background) |
| `torchvision.models.detection` | `maskrcnn_resnet50_fpn(_v2)` (instance) | COCO (80 classes) |
| `segmentation_models_pytorch` (smp) | U-Net, U-Net++, FPN, PSPNet, DeepLabV3+, … with any `timm` encoder | encoder pretrained on ImageNet |
| Hugging Face `transformers` | SegFormer, Mask2Former, UperNet | ADE20K, Cityscapes, COCO |
| Meta | SAM / SAM 2 (promptable, class-agnostic) | SA-1B (1.1B masks) |
| Medical | nnU-Net (self-configuring U-Net framework), MedSAM | many medical datasets |

Benchmark datasets: PASCAL VOC 2012, Oxford-IIIT Pet (used in our code notebook), Cityscapes, ADE20K, COCO-Stuff; medical: ISIC (skin lesions), BraTS (brain tumors), Kvasir-SEG (polyps).

---

# IV. **Object detection**

## IV.1 Problem setup
Output a **variable-size set** of detections $\{(\mathbf{b}_j, c_j, s_j)\}_{j=1}^{M}$: box $\mathbf{b}_j$, class $c_j$, confidence $s_j$. Box formats:
* corners $(x_1,y_1,x_2,y_2)$ — used by torchvision; center $(c_x,c_y,w,h)$ — used by YOLO; COCO json uses $(x,y,w,h)$ with $(x,y)$ the top-left corner.

Two difficulties distinguish detection from classification: (1) the number of objects $M$ is unknown, (2) we must *regress* continuous coordinates *and* classify.

**Intersection over Union** between boxes $A$ and $B$ is the basic geometric quantity:
$$
\text{IoU}(A,B)=\frac{|A\cap B|}{|A\cup B|},\qquad |A\cap B| = \max(0,\min(x_2^A,x_2^B)-\max(x_1^A,x_1^B))\cdot\max(0,\min(y_2^A,y_2^B)-\max(y_1^A,y_1^B)).
$$

<img src="https://raw.githubusercontent.com/d2l-ai/d2l-en/master/img/iou.svg" width="300">

*Figure: IoU is the ratio of the intersection area to the union area. Source: embedded from d2l.ai (Section 14.4).*

## IV.2 New components

### i) **Anchor boxes and box encoding**
Predicting raw coordinates is hard. Instead tile the image with reference boxes (**anchors**) at every feature-map location, with several scales and aspect ratios (e.g., 3 × 3 = 9 per location), and predict **offsets** relative to an anchor $(a_x,a_y,a_w,a_h)$ (center format):
$$
t_x=\frac{g_x-a_x}{a_w},\quad t_y=\frac{g_y-a_y}{a_h},\quad t_w=\log\frac{g_w}{a_w},\quad t_h=\log\frac{g_h}{a_h},
$$
where $\mathbf g$ is the matched ground-truth box. Decoding inverts this: $\hat g_x = a_x + a_w\hat t_x$, $\hat g_w = a_w e^{\hat t_w}$, etc. The log makes width/height errors scale-invariant.

**Anchor–ground-truth matching (label assignment):** an anchor is *positive* for the ground truth box with which its IoU $\ge 0.5$–$0.7$, *negative* (background) if max IoU $<0.3$–$0.4$, and ignored in between. **Anchor-free** detectors (FCOS, CenterNet, YOLOv8+) instead predict, for each feature-map location inside an object, the distances to the 4 box sides or the box center and size.

<img src="https://raw.githubusercontent.com/d2l-ai/d2l-en/master/img/anchor-label.svg" width="450">

*Figure: Assigning ground-truth boxes to anchor boxes by IoU. Source: embedded from d2l.ai (Section 14.4).*

### ii) **Non-maximum suppression (NMS)**
Many neighboring anchors fire on the same object. Greedy NMS per class:
1. Sort detections by score; 2. keep the top one $\mathbf b^*$; 3. discard all remaining boxes with $\text{IoU}(\mathbf b, \mathbf b^*) > \tau_{\text{nms}}$ (≈0.5); 4. repeat with the rest.

(`torchvision.ops.nms` / `batched_nms`.) Soft-NMS decays scores instead of discarding. DETR-style detectors avoid NMS entirely by set prediction with bipartite matching.

### iii) **Feature Pyramid Network (FPN)** (Lin et al., 2017)
Small objects need high-resolution features; large objects need deep semantic features. FPN builds a **top-down pathway with lateral connections**: $\;\mathbf P_\ell = \text{conv}_{3\times3}\big(\text{up}_{2\times}(\mathbf P_{\ell+1}) + \text{conv}_{1\times1}(\mathbf C_\ell)\big)$, giving semantically strong maps at every scale (strides 4, 8, 16, 32, 64). Objects are detected on the level matching their size. Used by Faster R-CNN-FPN, RetinaNet, Mask R-CNN, FCOS, YOLO (PAN-FPN variants).

### iv) **RoI Pooling / RoI Align**
Two-stage detectors must turn a box of arbitrary size on the feature map into a fixed-size tensor (e.g., $7\times7\times C$) for the head. RoI Pooling divides the box into a $7\times7$ grid and max-pools each cell (quantizing coordinates to integers). **RoI Align** avoids quantization by bilinearly sampling at exact fractional positions—important for small objects and masks (`torchvision.ops.roi_align`).

### v) **Box-aware data augmentation**
Geometric transforms must move the boxes too (flip: $x_1' = W - x_2$, $x_2' = W-x_1$); crops can cut boxes (clip, drop boxes that become too small). Detection-specific augmentations: **Mosaic** (4 images tiled into one, YOLOv4+), random scale "jitter", copy-paste. `torchvision.transforms.v2` handles boxes as `tv_tensors.BoundingBoxes`.

In [ ]:
# Snapshot: IoU, box encoding/decoding, and NMS (compare with torchvision.ops)
import torchvision.ops as ops

def box_iou(a, b):
    """a: (N,4), b: (M,4) in (x1,y1,x2,y2). Returns (N,M) IoU matrix."""
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    lt = torch.max(a[:, None, :2], b[None, :, :2])           # top-left of intersection
    rb = torch.min(a[:, None, 2:], b[None, :, 2:])           # bottom-right of intersection
    inter = (rb - lt).clamp(min=0).prod(dim=2)
    return inter / (area_a[:, None] + area_b[None, :] - inter)

def encode(gt, anchors):       # both (N,4) corners -> offsets (tx,ty,tw,th)
    g, a = ops.box_convert(gt, "xyxy", "cxcywh"), ops.box_convert(anchors, "xyxy", "cxcywh")
    return torch.cat([(g[:, :2] - a[:, :2]) / a[:, 2:], torch.log(g[:, 2:] / a[:, 2:])], dim=1)

def decode(t, anchors):
    a = ops.box_convert(anchors, "xyxy", "cxcywh")
    return ops.box_convert(torch.cat([a[:, :2] + t[:, :2] * a[:, 2:], a[:, 2:] * torch.exp(t[:, 2:])], 1), "cxcywh", "xyxy")

def nms(boxes, scores, thr=0.5):
    order, keep = scores.argsort(descending=True), []
    while order.numel() > 0:
        i = order[0]; keep.append(i.item())
        if order.numel() == 1: break
        iou = box_iou(boxes[i:i+1], boxes[order[1:]])[0]
        order = order[1:][iou <= thr]                         # drop boxes overlapping the kept one
    return torch.tensor(keep)

boxes = torch.tensor([[10, 10, 50, 50], [12, 12, 52, 52], [100, 100, 150, 160], [11, 9, 49, 51]], dtype=torch.float)
scores = torch.tensor([0.9, 0.8, 0.7, 0.95])
print("IoU matches torchvision:", torch.allclose(box_iou(boxes, boxes), ops.box_iou(boxes, boxes)))
print("our NMS keeps:", nms(boxes, scores).tolist(), "| torchvision NMS keeps:", ops.nms(boxes, scores, 0.5).tolist())
anchors = torch.tensor([[0, 0, 40, 40], [90, 90, 160, 150]], dtype=torch.float)
print("encode/decode round trip OK:", torch.allclose(decode(encode(boxes[[0, 2]], anchors), anchors), boxes[[0, 2]], atol=1e-4))

## IV.3 Architectures

### i) **Two-stage detectors: the R-CNN family**
1. **R-CNN** (Girshick et al., 2014): ~2000 region proposals from *selective search* (a non-learned algorithm) → crop and warp each → CNN features → SVM classifier + box regressor. Accurate but very slow (one CNN pass per proposal).
2. **Fast R-CNN** (2015): run the CNN **once** on the whole image; crop proposals from the *feature map* with RoI Pooling; one network does classification + box regression.
3. **Faster R-CNN** (Ren et al., 2015): replace selective search by a learned **Region Proposal Network (RPN)**: a small conv head sliding over the feature map predicts, for each anchor, an objectness score and box offsets. Stage 1 = RPN proposals (class-agnostic), stage 2 = RoI Align + head (class-specific scores + refined boxes). With FPN it remains a strong, accurate baseline.

<img src="https://raw.githubusercontent.com/d2l-ai/d2l-en/master/img/faster-rcnn.svg" width="450">

*Figure: Faster R-CNN. Source: embedded from d2l.ai (Section 14.8).*

### ii) **One-stage (single-shot) detectors**
Predict classes and boxes directly at every location/anchor, no proposal stage: faster, historically less accurate.
* **YOLO** (Redmon et al., 2016): divide the image into an $S\times S$ grid; the cell containing an object's center is responsible for it and predicts box + objectness + class probabilities. Later versions (v2–v11, YOLO-NAS, …) add anchors, multi-scale heads (FPN/PAN), better augmentation (Mosaic), and recently anchor-free heads and NMS-free training.
* **SSD** (Liu et al., 2016): anchors of different sizes on several feature maps of different resolution.
* **RetinaNet** (Lin et al., 2017): ResNet-FPN + dense anchors + **focal loss**. Showed that one-stage detectors were behind mainly because of the extreme foreground/background imbalance (~100k anchors, a handful of objects), which focal loss fixes.
* **FCOS** (Tian et al., 2019), **CenterNet** (Zhou et al., 2019): anchor-free.

### iii) **Set prediction with Transformers: DETR** (Carion et al., 2020)
CNN backbone + Transformer encoder–decoder outputs a fixed set of $N$ predictions; a **Hungarian (bipartite) matching** between predictions and ground truth defines the loss. No anchors, no NMS. Variants: Deformable DETR, DINO, RT-DETR (real-time).

In [ ]:
# Snapshot: a minimal single-scale, anchor-free, YOLO-style detection head on top of any backbone
class TinyDetectionHead(nn.Module):
    """For each cell of an S x S feature map predict: 1 objectness logit, 4 box values, K class logits."""
    def __init__(self, cin, n_classes):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv2d(cin, cin, 3, padding=1), nn.BatchNorm2d(cin), nn.ReLU(inplace=True))
        self.obj = nn.Conv2d(cin, 1, 1)
        self.box = nn.Conv2d(cin, 4, 1)          # (dx, dy) of center inside the cell, log(w), log(h) relative to stride
        self.cls = nn.Conv2d(cin, n_classes, 1)
    def forward(self, f):
        f = self.conv(f)
        return self.obj(f), self.box(f), self.cls(f)   # shapes (N,1,S,S), (N,4,S,S), (N,K,S,S)

backbone = nn.Sequential(SmallCNN().features)      # reuse the SmallCNN feature extractor: stride 8
head = TinyDetectionHead(128, n_classes=10)
obj, box, cls = head(backbone(torch.randn(2, 3, 96, 96)))
print("objectness", tuple(obj.shape), "| boxes", tuple(box.shape), "| classes", tuple(cls.shape))

In [ ]:
# Snapshot: torchvision Faster R-CNN with a new 2-class predictor (background + pedestrian).
# weights=None / weights_backbone=None keep this cell offline; use weights="DEFAULT" in practice.
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

det = fasterrcnn_resnet50_fpn(weights=None, weights_backbone=None)    # fasterrcnn_resnet50_fpn(weights="DEFAULT")
in_feats = det.roi_heads.box_predictor.cls_score.in_features
det.roi_heads.box_predictor = FastRCNNPredictor(in_feats, num_classes=2)  # class 0 is ALWAYS background
det.eval()
with torch.no_grad():
    out = det([torch.rand(3, 300, 400)])    # list of images (any size) -> list of dicts
print(out[0].keys())

## IV.4 Loss functions
Detection losses are **multi-task**: for each anchor/location $a$ with assigned target class $c_a^*$ ($0$ = background) and target box offsets $\mathbf t_a^*$,
$$
\mathcal{L}=\frac{1}{N_{\text{cls}}}\sum_a \mathcal{L}_{\text{cls}}(\hat{\mathbf p}_a, c^*_a)\;+\;\lambda\,\frac{1}{N_{\text{pos}}}\sum_a \mathbb{1}[c^*_a\ge 1]\;\mathcal{L}_{\text{box}}(\hat{\mathbf t}_a,\mathbf t^*_a).
$$
Box regression is only computed for **positive** anchors (background has no box).
* **Classification:** cross-entropy (Faster R-CNN; mini-batch sampled to 1:3 positive:negative to fight imbalance) or **focal loss** over all anchors (RetinaNet; $\alpha=0.25,\gamma=2$).
* **Box regression – Smooth L1 (Huber)** on each offset $d=\hat t-t^*$:
$$
\text{smooth}_{L_1}(d)=\begin{cases}0.5\,d^2/\beta & |d|<\beta\\ |d|-0.5\beta & \text{otherwise}\end{cases}
$$
(quadratic near 0 for smooth gradients, linear for large errors so outliers do not dominate).
* **IoU-based box losses** optimize the evaluation quantity directly and are scale-invariant: $\mathcal{L}_{\text{IoU}}=1-\text{IoU}$. When boxes do not overlap, IoU $=0$ has zero gradient, so **GIoU** (Rezatofighi et al., 2019) adds a penalty using the smallest enclosing box $C$:
$$
\text{GIoU}=\text{IoU}-\frac{|C\setminus(A\cup B)|}{|C|},\qquad \mathcal{L}_{\text{GIoU}}=1-\text{GIoU}\in[0,2].
$$
DIoU/CIoU add center-distance and aspect-ratio terms (YOLO family).
* **Objectness** (YOLO): BCE on "is there an object centered in this cell/anchor", target can be the IoU of the predicted box with the ground truth.
* **Two-stage total** (Faster R-CNN): $\mathcal{L}=\mathcal{L}^{\text{RPN}}_{\text{obj}}+\mathcal{L}^{\text{RPN}}_{\text{box}}+\mathcal{L}^{\text{head}}_{\text{cls}}+\mathcal{L}^{\text{head}}_{\text{box}}$ (torchvision returns these four terms as a dict in training mode).

## IV.5 Evaluation metrics: Average Precision (AP) and mAP
A detection counts as a **true positive** if it has the right class and $\text{IoU}\ge\tau$ with a ground-truth box *not already matched* by a higher-scoring detection; otherwise it is a false positive. Unmatched ground-truth boxes are false negatives.

For one class:
1. Sort all detections (across the whole test set) by confidence, descending.
2. Walk down the list; after the $r$-th detection compute $P_r=\frac{TP_r}{r}$ and $R_r=\frac{TP_r}{N_{\text{GT}}}$.
3. Make precision monotone (**interpolation**): $P^{\text{interp}}(R)=\max_{R'\ge R}P(R')$.
4. **AP = area under the interpolated PR curve**:
$$
\text{AP}=\sum_{r}(R_r-R_{r-1})\,P^{\text{interp}}(R_r)\quad(\text{all-point; COCO samples 101 recall points}).
$$

<img src="https://raw.githubusercontent.com/rafaelpadilla/Object-Detection-Metrics/master/aux_images/interpolated_precision_v2.png" width="420">

*Figure: Precision–recall curve (blue) and its interpolated version (red); AP is the area under the red curve. Source: embedded from github.com/rafaelpadilla/Object-Detection-Metrics.*

* **mAP** $=\frac{1}{K}\sum_{k=1}^K\text{AP}_k$ (mean over classes).
* **PASCAL VOC**: mAP@0.5 (IoU threshold 0.5).
* **COCO "AP"**: $\text{mAP@[.5:.95]}=\frac{1}{10}\sum_{\tau\in\{0.50,0.55,\dots,0.95\}}\text{mAP}@\tau$ — rewards precise localization; also AP$_{50}$, AP$_{75}$, AP$_S$/AP$_M$/AP$_L$ (small/medium/large objects) and **AR** (average recall).
* Libraries: `pycocotools`, `torchmetrics.detection.MeanAveragePrecision`.
* Speed matters too: report **FPS / latency** and **FLOPs** next to mAP.

# V. **Summary**

| | Binary classification | Multi-class classification | Semantic segmentation | Object detection |
|---|---|---|---|---|
| Output | 1 logit | $K$ logits | $K$ logits per pixel | per location: objectness, $K$ class logits, 4 box offsets |
| Final activation | sigmoid | softmax | softmax per pixel | sigmoid/softmax + box decoding + NMS |
| Loss | BCE (weighted, focal) | CE (label smoothing, focal) | pixel CE + Dice | cls CE/focal + Smooth-L1/GIoU (+ objectness) |
| Metrics | precision, recall, $F_1$, ROC-AUC, PR-AUC | top-$k$ acc., macro-$F_1$, confusion matrix, ECE | pixel acc., IoU/mIoU, Dice, HD95 | AP, mAP@0.5, mAP@[.5:.95] |
| Key components | augmentation, transfer learning | depthwise separable conv, inverted residual, SE, compound scaling | transposed conv, dilated conv, ASPP, U-Net skips | anchors + box encoding, NMS, FPN, RoI Align |
| Architectures | ResNet, EfficientNet, MobileNet, ConvNeXt | (same) | FCN, U-Net, DeepLabv3+, Mask R-CNN, SAM | Faster R-CNN, RetinaNet, SSD, YOLO, FCOS, DETR |
| XAI | saliency, IG, Grad-CAM, occlusion, LIME/SHAP | + contrastive Grad-CAM, error analysis | Seg-Grad-CAM, MC-dropout uncertainty | per-detection Grad-CAM, D-RISE, Eigen-CAM |

## References
* K. Murphy. *Probabilistic Machine Learning: An Introduction*. MIT Press, 2022, Ch. 14.
* A. Zhang, Z. Lipton, M. Li, A. Smola. *Dive into Deep Learning*, Ch. 8, 14. https://d2l.ai
* J. Long, E. Shelhamer, T. Darrell. Fully Convolutional Networks for Semantic Segmentation. CVPR 2015.
* O. Ronneberger, P. Fischer, T. Brox. U-Net. MICCAI 2015.
* L.-C. Chen et al. Rethinking Atrous Convolution (DeepLabv3), 2017; Encoder-Decoder with Atrous Separable Convolution (DeepLabv3+), ECCV 2018.
* S. Ren, K. He, R. Girshick, J. Sun. Faster R-CNN. NeurIPS 2015. K. He et al. Mask R-CNN. ICCV 2017.
* T.-Y. Lin et al. Feature Pyramid Networks, CVPR 2017; Focal Loss for Dense Object Detection (RetinaNet), ICCV 2017.
* J. Redmon et al. You Only Look Once. CVPR 2016. N. Carion et al. End-to-End Object Detection with Transformers (DETR). ECCV 2020.
* A. Howard et al. MobileNets, 2017; M. Sandler et al. MobileNetV2, CVPR 2018; J. Hu et al. Squeeze-and-Excitation Networks, CVPR 2018; M. Tan, Q. Le. EfficientNet, ICML 2019.
* R. Selvaraju et al. Grad-CAM. ICCV 2017. M. Sundararajan et al. Axiomatic Attribution (Integrated Gradients). ICML 2017. V. Petsiuk et al. Black-box Explanation of Object Detectors via Saliency Maps (D-RISE). CVPR 2021. K. Vinogradova et al. Towards Interpretable Semantic Segmentation via Gradient-weighted Class Activation Mapping. AAAI 2020. J. Adebayo et al. Sanity Checks for Saliency Maps. NeurIPS 2018.
* H. Rezatofighi et al. Generalized Intersection over Union. CVPR 2019.
* A. Kirillov et al. Segment Anything. ICCV 2023.